# Additional Practical Exercises — Data Engineering with Pandas
### Based on Class 4: Data Engineering with Pandas — Topic: Healthcare

These exercises use **3 files** (`consultas_medicas.csv`, `catalogo_tratamientos.csv`, `pacientes.json`) that simulate real data from a healthcare center, with **intentionally inserted data quality issues** (missing values, duplicates, malformed dates) so you can practice exactly what was covered in the original notebook: data ingestion, EDA, cleaning, `.dt`, `merge`, `groupby`/`agg`, `concat`, and a SQL query with `sqlite3`.

All data is synthetic and code-generated — it does not correspond to real patients, and all content is verifiable by running `.shape`, `.isna().sum()`, `.dtypes`, etc.

> 📎 Before running this notebook, make sure the following files are in the same folder: `consultas_medicas.csv`, `catalogo_tratamientos.csv`, `pacientes.json`

> **Note:** the source file names and column names remain in Spanish (`consultas_medicas.csv`, `id_paciente`, `fecha`, etc.) to match the original datasets. Only the instructions and comments in this notebook are in English.

## Data Structure

**`consultas_medicas.csv`** → `id_consulta, id_tratamiento, sesiones, fecha, id_paciente, modalidad`
- 164 rows (158 original + 6 exact duplicates inserted, simulating a double registration in the system)
- 23 missing values in `sesiones` (sessions)
- Some dates come in `YYYY/MM/DD` format instead of `YYYY-MM-DD` (invalid dates for `pd.to_datetime`)

**`catalogo_tratamientos.csv`** → `id_tratamiento, tratamiento, especialidad, costo`
- 9 rows (8 treatments + 1 exact duplicate, simulating a system sync error)
- 1 missing value in `costo` (cost) — the "Control Cardiológico" (Cardiology Checkup) row

**`pacientes.json`** → `id_paciente, nombre, ciudad, paciente_cronico`
- 20 patients (US cities: New York, Los Angeles, Chicago, Houston, Miami)
- 1 missing value in `ciudad` (city)

In [ ]:
import pandas as pd
import numpy as np
import sqlite3

pd.set_option('display.max_columns', None)

---
## 🧩 Exercise 1 — Ingestion, EDA, and Data Cleaning

**Goal:** practice `read_csv`, `read_json`, `.info()`, `.describe()`, `isna()`, `fillna()`, `dropna()`, `duplicated()`, and `drop_duplicates()`.

**Instructions:**

1. Load the three files (`consultas_medicas.csv` with `parse_dates=['fecha']`, `catalogo_tratamientos.csv`, `pacientes.json` with `orient='records'`).
2. Perform an initial audit of each DataFrame with `.info()` and `.describe()`.
3. Count missing values per column in each table with `.isna().sum()`.
4. Handle missing values with a justified criterion (as in the original notebook):
   - Missing `sesiones` in consultas → impute with `1` (logical minimum, assuming a single session) and convert to `int`.
   - Missing `costo` in the catalog → impute with the average cost of its `especialidad` (specialty).
   - Missing `ciudad` in patients → mark it as `'Unknown City'`.
5. Detect and remove duplicates:
   - Fully duplicated row in `catalogo_tratamientos` (use `duplicated(keep=False)` first to inspect it, then `drop_duplicates()`).
   - Exact duplicates in `consultas_medicas` (how many rows were there before and after?).
6. Convert `fecha` with `pd.to_datetime(..., errors='coerce')`, identify how many dates became `NaT`, and remove them with `dropna(subset=['fecha'])`.

**Ask yourself at the end:** how many rows did you start with in `consultas_medicas.csv`, and how many do you have left after cleaning? Document each decision with a comment, as the "Know-How" section of the original notebook requires.

In [ ]:
# 1. Load the files
df_consultas = None     # TODO: pd.read_csv('consultas_medicas.csv', parse_dates=['fecha'])
df_tratamientos = None  # TODO: pd.read_csv('catalogo_tratamientos.csv')
df_pacientes = None     # TODO: pd.read_json('pacientes.json', orient='records')


In [ ]:
# 2. Initial audit (.info() and .describe())


In [ ]:
# 3. Count missing values per column in each table


In [ ]:
# 4. Handling missing values


In [ ]:
# 5. Detecting and removing duplicates


In [ ]:
# 6. Converting and cleaning dates


---
## 🧩 Exercise 2 — Date Engineering + GroupBy/Agg

**Goal:** practice the `.dt` accessor, `groupby()` with `.agg()`, and `merge` (inner/left).

Use the cleaned consultations DataFrame from Exercise 1.

**Instructions:**

1. From the `fecha` column, extract: `anio` (year), `mes` (month), `nombre_mes` (month name), `dia_semana` (weekday), and `trimestre` (quarter) using `.dt`.
2. Merge `consultas_medicas` with `catalogo_tratamientos` (`merge`, `how='inner'`, key `id_tratamiento`), then with `pacientes` (`how='left'`, key `id_paciente`).
3. Create the column `costo_total = sesiones * costo`.
4. Answer the following using `groupby` + `.agg()`:
   - Total revenue, number of consultations, and average cost **per city**.
   - Sessions performed and revenue **per specialty and treatment**, sorted from highest to lowest revenue.
   - Compare revenue and average cost between `paciente_cronico` (`True` vs `False`).
   - Which `modalidad` (`Presencial`, `Virtual`, `Domicilio`) is used most? Which one generates the highest average revenue?
   - Which weekday (`dia_semana`) has the most scheduled consultations? (`value_counts()`)

In [ ]:
# 1. Date engineering with .dt


In [ ]:
# 2. Merge: consultas + catalog (inner) + patients (left)


In [ ]:
# 3. Create the costo_total = sesiones * costo column


In [ ]:
# 4. Revenue, number of consultations, and average cost per city


In [ ]:
# 4. Sessions and revenue per specialty and treatment (sorted descending)


In [ ]:
# 4. Comparison between paciente_cronico (True vs False)


In [ ]:
# 4. Most used modality and the one with the highest average revenue


In [ ]:
# 4. Weekday with the most consultations


---
## 🧩 Exercise 3 — Concat, Outer Join Audit, and SQL from Pandas

**Goal:** practice `pd.concat()`, `merge(how='outer', indicator=True)`, and `pd.read_sql()` with `sqlite3`.

**Instructions:**

1. Split the cleaned consultations DataFrame into two halves simulating two branches of the healthcare center (`Sede Norte` and `Sede Sur`, as in the original notebook), then merge them back together with `pd.concat(axis=0, ignore_index=True)`. Verify that the total row count matches.
2. Perform a `merge(how='outer', indicator=True)` between `consultas_medicas` and `catalogo_tratamientos` on `id_tratamiento`. Use `value_counts()` on the `_merge` column to audit: are there treatments in the catalog that were never scheduled? Are there consultations with an `id_tratamiento` that doesn't exist in the catalog?
3. Create an in-memory SQLite connection (`sqlite3.connect(':memory:')`), load the complete DataFrame (merged with patients) as the table `consultas_completo` using `.to_sql()`, and use `pd.read_sql()` to answer these 2 queries in pure SQL:
   - **Top 3 treatments by total revenue** (`GROUP BY`, `SUM`, `ORDER BY`, `LIMIT`).
   - **Revenue per month, only for months with more than 10 consultations** (`strftime('%Y-%m', fecha)`, `GROUP BY`, `HAVING`).
4. Close the connection with `conn.close()`.

In [ ]:
# 1. Split into two "branches" and concat again


In [ ]:
# 2. Outer merge with indicator=True for the audit


In [ ]:
# 3. In-memory SQLite connection + loading the table
conn = sqlite3.connect(':memory:')
# TODO: df_completo.to_sql('consultas_completo', conn, index=False, if_exists='replace')


In [ ]:
# 3a. Top 3 treatments by total revenue (SQL)
query_top_treatments = """
-- TODO: write your SQL query here
"""
# pd.read_sql(query_top_treatments, conn)


In [ ]:
# 3b. Revenue per month, only months with more than 10 consultations (SQL)
query_revenue_month = """
-- TODO: write your SQL query here
"""
# pd.read_sql(query_revenue_month, conn)


In [ ]:
# 4. Close the connection
conn.close()


---
## ✅ Self-Check Criteria (to review your own work)

- The final `.shape` of the cleaned `consultas_medicas` should be smaller than the original (due to removed duplicates and invalid dates).
- The cleaned `catalogo_tratamientos` should have 8 rows (one less than the original, because of the duplicate).
- No key column (`sesiones`, `costo`, `ciudad`) should have missing values after cleaning — confirm it with `.isna().sum()`.
- The sum of `costo_total` calculated with pandas should match the sum of `ingresos` (revenue) returned by the SQL query in Exercise 3 (it's the same information, calculated two different ways — a good way to verify you didn't make a mistake).

In [ ]:
# Free space for your final verification
